Fine-tune GPT or GPT-2 for creative story generation. in jupeter with comments 

1. Install Required Libraries

In [2]:
# Install required libraries
!pip install transformers datasets torch accelerate


In [3]:
!pip install --upgrade transformers


In [4]:
import transformers
print(transformers.__version__)


5.3.0


2. Import Libraries

In [5]:
# Import required modules
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments
from datasets import Dataset


3. Load GPT-2 Model and Tokenizer

In [6]:
# Load pretrained GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token

# Load pretrained GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

4. Prepare Training Dataset

In [7]:
# Example training data (you can add more stories)
stories = [
    "Once upon a time in a magical forest, a little rabbit discovered a glowing stone.",
    "In the year 3020, humans built cities in the clouds and robots became their friends.",
    "A lonely dragon found an ancient book that could change the fate of the kingdom.",
    "Deep under the ocean, a young explorer met a talking whale who knew many secrets.",
    "On a rainy evening, a mysterious door appeared in the middle of the street."
]

# Convert into dataset format
dataset = Dataset.from_dict({"text": stories})


5. Tokenize the Dataset

In [8]:
# Function to tokenize the dataset
def tokenize_function(example):
    
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    
    # Add labels for GPT-2 training
    tokens["labels"] = tokens["input_ids"].copy()
    
    return tokens

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

6. Set Training Arguments

In [9]:
from transformers import TrainingArguments

# Define training settings
training_args = TrainingArguments(
    
    output_dir="./story_model",   # Folder to save model
    
    num_train_epochs=3,           # Number of training rounds
    per_device_train_batch_size=2,
    
    save_steps=500,
    save_total_limit=2,
    
    logging_dir="./logs"
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


7. Create Trainer

In [10]:
# Trainer handles the training process

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)


8. Train the Model

In [11]:
# Start fine-tuning GPT-2
trainer.train()


c:\Users\samik\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:02<?, ?it/s]

TrainOutput(global_step=9, training_loss=2.9161353641086154, metrics={'train_runtime': 480.5794, 'train_samples_per_second': 0.031, 'train_steps_per_second': 0.019, 'total_flos': 489922560000.0, 'train_loss': 2.9161353641086154, 'epoch': 3.0})

9. Save the Fine-Tuned Model

In [12]:
# Save model and tokenizer
model.save_pretrained("./story_generator")
tokenizer.save_pretrained("./story_generator")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./story_generator\\tokenizer_config.json',
 './story_generator\\tokenizer.json')

10. Generate Creative Story

In [13]:
def generate_story(prompt):

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        inputs["input_ids"],
        max_length=120,
        num_return_sequences=1,
        temperature=0.9,          # more creativity
        top_k=50,
        top_p=0.95,
        do_sample=True,
        repetition_penalty=1.2    # reduce repetition
    )

    story = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return story

print(generate_story("Once upon a time"))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Once upon a time of crisis, the old and new will never be lost.
